# Phase 5 – Databricks + Olist Starter Notebook

In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
print('Setup complete')

In [ ]:
display(dbutils.fs.ls('/FileStore/olist/'))

In [ ]:
orders = spark.read.csv('/FileStore/olist/olist_orders_dataset.csv', header=True, inferSchema=True)
order_items = spark.read.csv('/FileStore/olist/olist_order_items_dataset.csv', header=True, inferSchema=True)
customers = spark.read.csv('/FileStore/olist/olist_customers_dataset.csv', header=True, inferSchema=True)
products = spark.read.csv('/FileStore/olist/olist_products_dataset.csv', header=True, inferSchema=True)
payments = spark.read.csv('/FileStore/olist/olist_order_payments_dataset.csv', header=True, inferSchema=True)

In [ ]:
display(orders)
display(customers)
display(order_items)
orders.printSchema()

In [ ]:
orders.select([count(when(col(c).isNull(), c)).alias(c) for c in orders.columns]).show()
print('Orders:', orders.count())
print('Customers:', customers.count())

In [ ]:
orders_customers = orders.join(customers, 'customer_id', 'inner')
display(orders_customers)

In [ ]:
fact_orders = order_items.join(orders, 'order_id').join(customers, 'customer_id').join(payments, 'order_id')
display(fact_orders)

## Tasks (Students Implement Below)

In [ ]:
# Task 1: Top 3 Customers per City
# TODO
from pyspark.sql.functions import sum
from pyspark.sql.window import Window
from pyspark.sql.functions import rank

df = customers.join(orders, "customer_id").join(payments, "order_id")
customer_spend = df.groupBy("customer_city", "customer_id").agg(sum("payment_value").alias("total_spend"))
windowSpec = Window.partitionBy("customer_city").orderBy(customer_spend["total_spend"].desc())

result = customer_spend.withColumn("rank", rank().over(windowSpec)).filter("rank <= 3")
display(result)

In [ ]:
# Task 2: Running Total of Sales
# TODO
from pyspark.sql.functions import to_date, sum
from pyspark.sql.window import Window

sales = payments.join(orders, "order_id").withColumn("date", to_date("order_purchase_timestamp"))

daily_sales = sales.groupBy("date").agg(sum("payment_value").alias("daily_sales"))

windowSpec = Window.orderBy("date")

result = daily_sales.withColumn("running_total",sum("daily_sales").over(windowSpec))

display(result)

In [ ]:
# Task 3: Top Products per Category
# TODO
from pyspark.sql.functions import dense_rank

df = order_items.join(products, "product_id")

product_sales = df.groupBy("product_category_name", "product_id") .agg(sum("price").alias("total_sales"))

windowSpec = Window.partitionBy("product_category_name").orderBy(product_sales["total_sales"].desc())

result = product_sales.withColumn("rank", dense_rank().over(windowSpec))

display(result)

In [ ]:
# Task 4: Customer Lifetime Value
# TODO
clv = payments.join(orders, "order_id").groupBy("customer_id").agg(sum("payment_value").alias("total_spend"))

display(clv)

In [ ]:
# Task 5: Customer Segmentation
# TODO
from pyspark.sql.functions import when

segmented = clv.withColumn("segment",
    when(clv.total_spend > 10000, "Gold").when((clv.total_spend >= 5000) & (clv.total_spend <= 10000), "Silver").otherwise("Bronze")
)

display(segmented)
segment_count = segmented.groupBy("segment").count()
display(segment_count)

In [ ]:
# Task 6: Final Reporting Table
# TODO
orders_count = orders.groupBy("customer_id").count().withColumnRenamed("count", "total_orders")

final_df = segmented.join(customers, "customer_id").join(orders_count, "customer_id").select("customer_id", "customer_city", "total_spend", "segment", "total_orders")

display(final_df)